# Lezione 04 - Design Pattern per l'Uso degli Strumenti

In questa lezione imparerai il design pattern **Tool Use** per agenti AI usando il Microsoft Agent Framework (Python). Tratteremo:

- Definire strumenti funzione con il decoratore `@tool` e parametri tipizzati
- Fornire schemi degli strumenti in modo che il modello sappia cosa fa ogni strumento
- Controllare l'esecuzione dello strumento con `approval_mode`
- Restituire **output strutturato** tramite modelli Pydantic e `response_format`

Lo scenario è un **agente di prenotazione viaggi** che può cercare destinazioni, controllare disponibilità e recuperare informazioni sui voli.


## Configurazione


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Definire Strumenti con il Decoratore @tool

Il decoratore `@tool` trasforma una funzione Python semplice in uno strumento che un agente può chiamare.
Punti chiave:

- La **docstring** diventa la descrizione dello strumento che il modello vede.
- Le **annotazioni di tipo** (incluso `Annotated` con descrizioni) definiscono lo schema dello strumento.
- `approval_mode` controlla se l'utente deve approvare ogni chiamata prima che venga eseguita.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Creazione di un agente con più strumenti

Passa tutti e tre gli strumenti al client in modo che il modello possa invocare quelli che gli servono per rispondere alla domanda dell'utente.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Output Strutturato con Strumenti

Impostando `response_format` su un modello Pydantic, l'agente è costretto a restituire un oggetto JSON ben tipizzato invece di testo libero. Questo è utile quando il codice a valle deve consumare il risultato in modo programmato.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Modelli di Approvazione degli Strumenti

Il parametro `approval_mode` su `@tool` controlla se le chiamate allo strumento richiedono l'approvazione umana prima dell'esecuzione:

| Modalità | Comportamento |
|---|---|
| `"never_require"` | Lo strumento viene eseguito automaticamente — non è necessaria conferma dall'utente. |
| `"always_require"` | Ogni chiamata deve essere approvata dall'utente prima dell'esecuzione. |

Usa `"always_require"` per strumenti che hanno effetti collaterali (ad es. prenotare un volo, addebitare una carta di credito) così una persona resta coinvolta nel processo.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Sommario

In questa lezione hai imparato a:

1. **Definire strumenti** usando il decoratore `@tool` con parametri tipizzati e docstring che fungono da schema dello strumento.
2. **Comporre più strumenti** in modo che l'agente possa chiamarli in sequenza per rispondere a query complesse.
3. **Restituire output strutturato** passando un modello Pydantic come `response_format`.
4. **Controllare l'approvazione dello strumento** con `approval_mode` per mantenere un supervisore umano nelle operazioni sensibili.

Questi schemi costituiscono le basi per costruire agenti affidabili e pronti per la produzione che possono interagire in sicurezza con sistemi esterni.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
